In [6]:
import pandas as pd
import numpy as np

churn_df = pd.read_csv('../data/churn_labels.csv')
account = pd.read_csv('../data/account.csv', sep=';')
client = pd.read_csv('../data/client.csv', sep=';')
disp = pd.read_csv('../data/disp.csv', sep=';')
loan = pd.read_csv('../data/loan.csv', sep=';')
card = pd.read_csv('../data/card.csv', sep=';')
order = pd.read_csv('../data/order.csv', sep=';')
district = pd.read_csv('../data/district.csv', sep=';', header=None)
district.columns = ['district_id','district_name','region','n_inhabitants',
    'n_muni_lt499','n_muni_500_1999','n_muni_2000_9999','n_muni_gt10000',
    'n_cities','urban_ratio','avg_salary','unemp_95','unemp_96',
    'n_entrepreneurs_per1000','crimes_95','crimes_96']

account['date_parsed'] = pd.to_datetime(account['date'], format='%y%m%d')
trans = pd.read_csv('../data/trans.csv', sep=';')
trans['date_parsed'] = pd.to_datetime(trans['date'], format='%y%m%d')
trans_obs = trans[trans['date_parsed'] <= pd.Timestamp('1997-12-31')]

C:\Users\user\AppData\Local\Temp\ipykernel_43768\2025626064.py:18: DtypeWarning: Columns (0: bank) have mixed types. Specify dtype option on import or set low_memory=False.
  trans = pd.read_csv('../data/trans.csv', sep=';')


In [7]:
tx_feat = trans_obs.groupby('account_id').agg(
    tx_count=('trans_id','count'),
    tx_amount_mean=('amount','mean'),
    tx_amount_std=('amount','std'),
    balance_mean=('balance','mean'),
    balance_min=('balance','min'),
    balance_last=('balance', lambda x: x.iloc[-1])
).reset_index()

# tenure: years active in observation window
tx_feat['tx_count_per_year'] = tx_feat['tx_count'] / 5

In [8]:
acc_feat = account[['account_id','district_id','frequency']].copy()
acc_feat['tenure_days'] = (pd.Timestamp('1997-12-31') - account['date_parsed']).dt.days

In [9]:
loan_feat = loan.groupby('account_id').size().reset_index(name='has_loan')
loan_feat['has_loan'] = 1

card_feat = disp[disp['account_id'].isin(card['disp_id'])][['account_id']].drop_duplicates()
card_feat['has_card'] = 1

order_feat = order.groupby('account_id').agg(n_orders=('order_id','count'), order_amount_sum=('amount','sum')).reset_index()

In [11]:
features = churn_df[['account_id','churned']].copy()
features = features.merge(acc_feat, on='account_id', how='left')
features = features.merge(tx_feat, on='account_id', how='left')
features = features.merge(loan_feat[['account_id','has_loan']], on='account_id', how='left')
features = features.merge(card_feat, on='account_id', how='left')
features = features.merge(order_feat, on='account_id', how='left')
district = district[pd.to_numeric(district['district_id'], errors='coerce').notna()]
district['district_id'] = district['district_id'].astype(int)
features['district_id'] = features['district_id'].astype(int)
features = features.merge(district, on='district_id', how='left')

# fill NaN for accounts with no loan/card/orders — absence is meaningful, not missing data
features[['has_loan','has_card']] = features[['has_loan','has_card']].fillna(0)
features[['n_orders','order_amount_sum']] = features[['n_orders','order_amount_sum']].fillna(0)

print(features.shape)
features.isnull().sum()

(4057, 31)


account_id                 0
churned                    0
district_id                0
frequency                  0
tenure_days                0
tx_count                   0
tx_amount_mean             0
tx_amount_std              0
balance_mean               0
balance_min                0
balance_last               0
tx_count_per_year          0
has_loan                   0
has_card                   0
n_orders                   0
order_amount_sum           0
district_name              0
region                     0
n_inhabitants              0
n_muni_lt499               0
n_muni_500_1999            0
n_muni_2000_9999           0
n_muni_gt10000             0
n_cities                   0
urban_ratio                0
avg_salary                 0
unemp_95                   0
unemp_96                   0
n_entrepreneurs_per1000    0
crimes_95                  0
crimes_96                  0
dtype: int64

In [12]:
print(features.shape)
features.to_csv('../data/features.csv', index=False)

(4057, 31)
